# Code Execution and the Files API

Two features that shine together:

**Files API** — upload a file once (image, PDF, CSV, ...) and get a `file_id` you reference in later messages, instead of base64-inlining it every request. Better for large or repeatedly-used files.

**Code execution** — a **server-side** tool (no implementation from you) that lets Claude run **Python/bash in an isolated Docker sandbox**. Key traits:
- isolated container, **no network access** (can't call external APIs)
- Claude can run code **multiple times** in one turn, iterating
- results are captured and interpreted by Claude for its final answer

**Why together:** since the sandbox has no network, the **Files API is how data gets in and out** — upload a CSV, reference it via a `container_upload` block, let Claude analyze it and generate outputs (plots) you then download by `file_id`.

> **Verified for this notebook:** runs on `claude-haiku-4-5-20251001`, which supports tool version **`code_execution_20250825`**. Requires two beta headers — **`code-execution-2025-08-25`** and **`files-api-2025-04-14`** (the lesson text's `code_execution_20250522` is the legacy Python-only version).

## Setup — client (beta headers), Files API helpers

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
from pathlib import Path
from anthropic import Anthropic
from anthropic.types import Message

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


# ---- Files API helpers ----
def upload(file_path):
    path = Path(file_path)
    mime_type_map = {
        ".pdf": "application/pdf", ".txt": "text/plain", ".md": "text/plain",
        ".py": "text/plain", ".csv": "text/csv", ".json": "application/json",
        ".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
    }
    mime_type = mime_type_map.get(path.suffix.lower())
    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {path.suffix}")
    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(path.name, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)
    file_content.write_to_file(filename or get_metadata(id).filename)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Upload the CSV

`streaming.csv` holds 500 streaming-service users: subscription tier, viewing habits, support interactions, monthly cost, and whether they `Churned`. Upload once, get a `file_id`.

In [2]:
SECTION = os.path.join(os.path.dirname(find_dotenv()), "01-building-with-the-claude-api", "06-features")

file_metadata = upload(os.path.join(SECTION, "streaming.csv"))
print("file id:", file_metadata.id)
file_metadata

file id: file_011CcyERQKoxy2FPD7B6sBtz


FileMetadata(id='file_011CcyERQKoxy2FPD7B6sBtz', created_at=datetime.datetime(2026, 7, 13, 2, 1, 18, 533000, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, scope=None)

## Ask Claude to analyze it with code execution

The message pairs a text instruction with a **`container_upload`** block referencing the `file_id`. We pass the `code_execution` tool; Claude writes Python, runs it in the sandbox (possibly several times), and produces a plot.

The note about a "clean slate" each execution reflects the sandbox: with `code_execution_20250825`, each run starts fresh, so Claude must re-import/redeclare every time. (This can take a while — it's doing real data science.)

In [3]:
messages = []
add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate.
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

response = chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])
print("stop_reason:", response.stop_reason)
print("block types:", [b.type for b in response.content])

stop_reason: end_turn
block types: ['text', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'text_editor_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'bash_code_execution_tool_result', 'server_tool_use', 'bash_code_execution_tool_result', 'text', 'server_tool_use', 'bash_code_execution

## Read the response

Code-execution responses interleave several block types:
- **`text`** — Claude's analysis/explanation
- **`server_tool_use`** — the actual code Claude chose to run
- **`code_execution_tool_result`** — stdout/stderr and any output files

This helper walks them so you can see the code and the narrative.

In [4]:
def render_code_execution(message):
    for block in message.content:
        if block.type == "text":
            print("=== TEXT ===")
            print(block.text)
        elif block.type == "server_tool_use":
            print("=== CODE CLAUDE RAN ===")
            code = block.input.get("code") if isinstance(block.input, dict) else block.input
            print(code)
        elif block.type == "code_execution_tool_result":
            print("=== EXECUTION RESULT ===")
            content = getattr(block, "content", None)
            stdout = getattr(content, "stdout", None)
            if stdout:
                print(stdout[:1000])
        print()


render_code_execution(response)

=== TEXT ===
I'll run a detailed analysis to determine major drivers of churn. Let me start by exploring the data.

=== CODE CLAUDE RAN ===
None


=== TEXT ===
Now let me conduct a comprehensive analysis of the churn drivers:

=== CODE CLAUDE RAN ===
None


=== CODE CLAUDE RAN ===
None


=== TEXT ===
Excellent! Now let me create a comprehensive visualization summarizing these findings:

=== CODE CLAUDE RAN ===
None


=== CODE CLAUDE RAN ===
None


=== TEXT ===
Let me fix that issue:

=== CODE CLAUDE RAN ===
None


=== CODE CLAUDE RAN ===
None


=== CODE CLAUDE RAN ===
None


=== TEXT ===
Perfect! Now let me copy the visualizations to the output directory and create a final summary report:

=== CODE CLAUDE RAN ===
None


=== TEXT ===
Now let me create a detailed executive summary report:

=== CODE CLAUDE RAN ===
None


=== CODE CLAUDE RAN ===
None


=== TEXT ===
Perfect! Now let me create one final summary file listing all outputs:

=== CODE CLAUDE RAN ===
None


=== TEXT ===
Perfect! L

## Download generated files

When Claude creates a plot, it lands in the container and shows up as a `code_execution_output` with a `file_id`. Pull those IDs out of the response and download them via the Files API (rather than hardcoding an ID). Saved into `code-exec-output/`.

In [5]:
def generated_file_ids(message):
    ids = []
    for block in message.content:
        if block.type == "code_execution_tool_result":
            content = getattr(block, "content", None)
            for item in (getattr(content, "content", None) or []):
                if getattr(item, "type", None) == "code_execution_output":
                    fid = getattr(item, "file_id", None)
                    if fid:
                        ids.append(fid)
    return ids


out_dir = os.path.join(SECTION, "code-exec-output")
os.makedirs(out_dir, exist_ok=True)

ids = generated_file_ids(response)
print("generated files:", ids)

for fid in ids:
    name = get_metadata(fid).filename or f"{fid}.bin"
    download_file(fid, os.path.join(out_dir, name))
    print("downloaded:", name)

generated files: []


In [6]:
# Show the first generated image inline, if any
from IPython.display import Image as IPyImage

pngs = [f for f in os.listdir(out_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))] if os.path.isdir(out_dir) else []
IPyImage(filename=os.path.join(out_dir, pngs[0])) if pngs else print("No image files generated.")

No image files generated.


## Beyond data analysis

The Files-API + code-execution combo delegates any computational task to Claude while you control inputs/outputs by `file_id`: image processing, document parsing/transformation, math/modeling, custom report generation. Claude becomes a coding assistant that can **actually run and iterate** on solutions in a sandbox.

> **Housekeeping:** uploaded files persist on Anthropic's side — use `list_files()` / `delete_file(id)` to manage them. The sandbox has no network, so the Files API is the only data in/out path.

That's the last lesson of **Features of Claude** — next is the section quiz.